# Data Creation Playground

Full multi-depth pruning pipeline. G = Gemma-4 (Colab GPU), D = Gemini Flash Lite (cloud).

**Before running:** Enable GPU runtime → Runtime → Change runtime type → T4 GPU (or A100).

In [ ]:
!git clone https://github.com/avrymi-asraf/reasoning-pruning.git
%cd reasoning-pruning
# Gemma 4 requires transformers from git main — not yet in a stable PyPI release
!pip install -q "git+https://github.com/huggingface/transformers.git" "accelerate>=0.34.0" "datasets>=4.8.5" "torchvision>=0.27.0" "pyyaml>=6.0.2"

In [ ]:
import os
from google.colab import userdata

os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
os.environ["GEMINI_API_KEY"] = userdata.get("GEMINI_API_KEY")

In [ ]:
import sys
sys.path.insert(0, "src")

from pathlib import Path
from dataclasses import replace

from reasoning_pruning.data_creation import (
    load_data_creation_config,
    load_questions,
    build_pt_dataset,
    build_rows_for_question,
    format_context,
    split_reasoning_units,
)
from reasoning_pruning.clients import TransformersGenerator, GeminiDecisionModel

print("Imports OK")

In [ ]:
# Downloads ~5GB from Hub — takes 1-2 min on first run
generator = TransformersGenerator(
    source_model="avreymi/gemma-4-E2B-it-reasoning-pruning",
    generation_config={"max_new_tokens": 512, "temperature": 0.7, "do_sample": True},
)
print("G ready:", generator.source_model)

In [ ]:
decision_model = GeminiDecisionModel(
    decision_model="gemini-2.0-flash-lite",
    prompt_version="conservative-skip-v1",
    prompts_dir="prompts",
)
print("D ready:", decision_model.decision_model)

In [ ]:
# Load config from YAML, then override depth/limit for quick playground runs
config = load_data_creation_config(Path("configs/data/dataset_builder_gsm8k_100_gemma4.yaml"))
config = replace(config, max_pruning_depth=4, max_examples_per_question=4)

# --- Questions to experiment with ---
# Use any of these, or replace with your own
questions = [
    "A notebook costs $5. Mina buys 4 notebooks. What is the total cost?",
    "If a train travels 60 mph for 2.5 hours, how far does it travel?",
    "A store has 48 apples. It sells 3/4 of them. How many are left?",
]

print(f"Config: max_depth={config.max_pruning_depth}, G={config.generator['model_id']}")
print(f"Questions: {len(questions)}")

In [ ]:
def show_rows(rows: list[dict]) -> None:
    if not rows:
        print("No rows generated — D found no safe removals at any depth")
        return
    for row in rows:
        sep = "=" * 70
        print(f"\n{sep}")
        print(f"  Depth {row['pruning_depth']}")
        print(f"{sep}")
        print(f"\n[GENERATED UNITS]")
        for i, u in enumerate(row["generated_units"]):
            marker = "  ✗" if row["metadata"]["removed_start_index"] <= i <= row["metadata"]["removed_end_index"] else "   "
            print(f"{marker} {i}: {u}")
        print(f"\n[REMOVED] indices {row['metadata']['removed_start_index']}–{row['metadata']['removed_end_index']}")
        print(f"  Reason: {row['metadata']['decision_reason']}")
        print(f"\n[INPUT_X]")
        for line in row["input_x"].splitlines():
            print(f"  {line}")
        print(f"\n[TARGET_Y]")
        print(f"  {row['target_y']}")
    print(f"\n  → {len(rows)} training row(s) from this question")

In [ ]:
# Run on a single question to inspect each depth in detail
rows = build_rows_for_question(
    question=questions[0],
    generator=generator,
    decision_model=decision_model,
    config=config,
)
show_rows(rows)

In [ ]:
# Run all questions and get a summary
all_rows = build_pt_dataset(
    questions=questions,
    generator=generator,
    decision_model=decision_model,
    config=config,
)

print(f"Total rows: {len(all_rows)}")
print(f"Rows per question: {len(all_rows) / len(questions):.1f} avg")
print(f"Depths seen: {sorted(set(r['pruning_depth'] for r in all_rows))}")
print()
for row in all_rows:
    q_preview = row["question"][:60]
    removed = " | ".join(row["metadata"]["removed_span"])
    print(f"[d={row['pruning_depth']}] {q_preview}...")
    print(f"  removed: {removed[:80]}")
    print(f"  target : {row['target_y'][:80]}")